# Part I: Language Model Training and Comparison

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import numpy as np
import pandas as pd
from datasets import load_dataset
from tqdm import tqdm
import time
import math
import re
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Embedding, Dense, Input, LayerNormalization, MultiHeadAttention
from tensorflow.keras.models import Model

c:\Users\jenni\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("stanfordnlp/imdb")
train_lines = dataset["train"].to_pandas()

In [3]:
train_lines = train_lines['text'].dropna().astype(str)

In [4]:
tokenizer = Tokenizer()
tokenizer.fit_on_texts(train_lines)
sequences = tokenizer.texts_to_sequences(train_lines)

In [5]:
max_len = 300
sequences = pad_sequences(sequences, maxlen=max_len, padding='post')

In [6]:
vocab_size = len(tokenizer.word_index) + 1

In [7]:
d_model = 128 # Embedding and model dimension
num_heads = 4 # Number of attention heads
ff_dim = 256 # Hidden dimension of feed-forward layer
num_decoder_layers = 2
dropout_rate = 0.1

In [8]:
class PositionalEmbedding(tf.keras.layers.Layer):
    def __init__(self, max_len, d_model, **kwargs):
        super().__init__(**kwargs)
        self.token_emb = Embedding(input_dim=max_len, output_dim=d_model)
        self.position_emb = Embedding(input_dim=max_len, output_dim=d_model)

    def call(self, x):
        # x has shape (batch_size, seq_len)
        max_len = tf.shape(x)[1]
        positions = tf.range(start=0, limit=max_len, delta=1)
        # (batch_size, seq_len, d_model)
        token_embeddings = self.token_emb(x)
        position_embeddings = self.position_emb(positions)
        return token_embeddings + position_embeddings


In [9]:
def decoder_layer(x, d_model, num_heads, ff_dim, dropout_rate):
    # (Masked) Self-attention
    attn1 = MultiHeadAttention(num_heads=num_heads, key_dim=d_model,dropout=dropout_rate)(x, x)
    out1 = LayerNormalization(epsilon=1e-6)(x + attn1)
    
    # Feed-forward
    ffn = Dense(ff_dim, activation='relu')(out1)
    ffn = Dense(d_model)(ffn)
    out3 = LayerNormalization(epsilon=1e-6)(out1 + ffn)
    return out3

In [10]:
def build_decoder(inp, num_layers,d_model,num_heads,ff_dim,dropout_rate):
    x = inp
    for _ in range(num_layers):
        x = decoder_layer(x, d_model, num_heads, ff_dim, dropout_rate)
    return x

In [11]:
# Transformer (Decoder Only because not a machine translation)
def build_transformer(vocab_size, d_model,num_heads,ff_dim,
                      num_decoder_layers, dropout_rate=0.1):

    inputs = Input(shape=(None,), name="inputs")
    
    # Embedding
    emb = Embedding(input_dim=vocab_size, output_dim=d_model)(inputs)
    
    output = build_decoder(emb, num_layers=num_decoder_layers,
                               d_model=d_model,num_heads=num_heads,ff_dim=ff_dim,dropout_rate=dropout_rate)
    
    # ----- FINAL LINEAR LAYER -----
    final_output = Dense(vocab_size, activation="softmax")(output)
    
    # Create the model
    model = Model(inputs, final_output)
    return model

In [12]:
model = build_transformer(vocab_size, d_model=d_model,num_heads=num_heads, 
                          ff_dim=ff_dim, num_decoder_layers=num_decoder_layers,
                          dropout_rate=dropout_rate)

model.compile(optimizer="adam", loss="crossentropy", 
              metrics=["accuracy"])

In [13]:
model.summary()

Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ inputs (InputLayer) │ (None, None)      │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ embedding           │ (None, None, 128) │ 11,338,624 │ inputs[0][0]      │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, None, 128) │    263,808 │ embedding[0][0],  │
│ (MultiHeadAttentio… │                   │            │ embedding[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, None, 128) │          0 │ embedding[0][0],  │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, None, 128) │        256 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, None, 256) │     33,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, None, 128) │     32,896 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, None, 128) │          0 │ layer_normalizat… │
│                     │                   │            │ dense_1[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, None, 128) │        256 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, None, 128) │    263,808 │ layer_normalizat… │
│ (MultiHeadAttentio… │                   │            │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, None, 128) │          0 │ layer_normalizat… │
│                     │                   │            │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, None, 128) │        256 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, None, 256) │     33,024 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_3 (Dense)     │ (None, None, 128) │     32,896 │ dense_2[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, None, 128) │          0 │ layer_normalizat… │
│                     │                   │            │ dense_3[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, None, 128) │        256 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_4 (Dense)     │ (None, None,      │ 11,427,207 │ layer_normalizat… │
│                     │ 88583)            │            │                   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,426,311 (89.36 MB)

 Trainable params: 23,426,311 (89.36 MB)

 Non-trainable params: 0 (0.00 B)

In [14]:
X = sequences[:, :-1] 
y = sequences[:, 1:]

In [15]:
start_train = time.time()
history = model.fit(x = X, y = y, batch_size= 32, epochs= 5)
end_train = time.time()

Epoch 1/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 2754s 4s/step - accuracy: 0.4407 - loss: 4.1677
Epoch 2/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 3220s 4s/step - accuracy: 0.4721 - loss: 3.4932
Epoch 3/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 3335s 4s/step - accuracy: 0.4846 - loss: 3.2509
Epoch 4/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 2594s 3s/step - accuracy: 0.4926 - loss: 3.0715
Epoch 5/5
782/782 ━━━━━━━━━━━━━━━━━━━━ 2533s 3s/step - accuracy: 0.4983 - loss: 2.9262


In [16]:
print("training time: ", end_train - start_train)

training time:  14435.228889465332


In [74]:
print(history.history['loss'])

[4.167747497558594, 3.4931960105895996, 3.2509336471557617, 3.0714592933654785, 2.9262380599975586]


In [62]:
def generate_text(model, tokenizer, start_text, max_len):
    
    start_seq = tokenizer.texts_to_sequences([start_text])[0]
    
    generated = list(start_seq)
    words = start_text.split()
    
    for _ in range(max_len):
        current_seq = pad_sequences([generated], maxlen = 300, padding='post')
        preds = model.predict(current_seq, verbose=0)
        next_token = np.argmax(preds[0, len(generated)-1, :])
        
        if next_token == 0:
            break
            
        reverse_map = {v: k for k, v in tokenizer.word_index.items()}
        reverse_map[0] = ''
        next_word = reverse_map.get(next_token, '')
        words.append(next_word)
        
        generated.append(next_token)
        
        if next_word == '.':
            break
    
    return ' '.join(words)

In [ ]:
# Tried the normal generate text but the generated text is the same as start text
start_time = time.time()
result = generate_text(model, tokenizer, "This movie", 20)
end_time = time.time()

print(f"\nGenerated Text: {result}")


Generated Text: This movie


In [69]:
print(f"Inference time: {end_time - start_time:.2f}s")

Inference time: 0.19s


In [65]:
def generate_text_with_temperature(model, tokenizer, start_text, max_len, temperature=0.8):
    start_seq = tokenizer.texts_to_sequences([start_text])[0]   
    
    generated = list(start_seq)
    words = start_text.split()
    
    for _ in range(max_len):
        current_seq = pad_sequences([generated], maxlen=300, padding='post')
        preds = model.predict(current_seq, verbose=0)
        next_token_logits = preds[0, len(generated)-1, :].copy()
        
        # Apply temperature
        next_token_logits = next_token_logits / temperature
        
        # Convert to probabilities
        exp_logits = np.exp(next_token_logits - np.max(next_token_logits))
        probs = exp_logits / np.sum(exp_logits)
        
        # Sample from the distribution (not just argmax!)
        next_token = np.random.choice(len(probs), p=probs)
        
        # If we somehow got padding token, try again with next best
        if next_token == 0:
            probs[0] = 0
            probs = probs / np.sum(probs)
            next_token = np.random.choice(len(probs), p=probs)
        
        # Convert token to word
        reverse_map = {v: k for k, v in tokenizer.word_index.items()}
        reverse_map[0] = ''
        next_word = reverse_map.get(next_token, '')
        
        # print(f"Step {i}: '{next_word}'")
        
        words.append(next_word)
        generated.append(next_token)
        
        # Stop at sentence end
        if next_word in '.':
            break
    
    return ' '.join(words)


In [66]:
start_inf = time.time()
result = generate_text_with_temperature(model, tokenizer, "This movie", 20, temperature=0.8)
print(f"\nGenerated: {result}")
end_inf = time.time()


Generated: This movie scribble fricken' swishing smashmouth garner helping slashes she´s sizemore luxurious focus him unfussy wrangles siesta klux toy cabin intruders'' escrow


In [67]:
print(f"Inference time: {end_inf - start_inf:.2f}s")

Inference time: 4.42s
